# 📋 Notebook 4: Business Recommendations

**Purpose:** Translate statistical results into stakeholder-ready recommendations.
This is the output that goes to your product manager, design team, or executive.

> A great analyst doesn't just report p-values. They tell the business what to do next and why.


In [1]:
import sys
sys.path.append('../src')
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from ab_tests import (
    select_and_run_test, assess_practical_significance,
    check_sample_ratio_mismatch, detect_novelty_effect, generate_recommendation
)

# Load data
s1 = pd.read_csv('../data/scenario1_ship_it.csv')
s2 = pd.read_csv('../data/scenario2_do_not_ship.csv')
s3 = pd.read_csv('../data/scenario3_inconclusive.csv')
s1_daily = pd.read_csv('../data/scenario1_daily_rates.csv')
s3_daily = pd.read_csv('../data/scenario3_daily_rates.csv')
print("Data loaded. Generating recommendation memos...")

Data loaded. Generating recommendation memos...


---
## Scenario 1: CTA Button Test → SHIP IT

In [2]:
s1c, s1v = s1[s1.group=='control'], s1[s1.group=='variant']

result1   = select_and_run_test(metric_type='proportion',
    control_conversions=int(s1c.converted.sum()), control_visitors=len(s1c),
    variant_conversions=int(s1v.converted.sum()), variant_visitors=len(s1v))
srm1      = check_sample_ratio_mismatch(len(s1c), len(s1v))
novelty1  = detect_novelty_effect(s1_daily['control_rate'], s1_daily['variant_rate'])
practical1 = assess_practical_significance(result1['relative_lift_pct'], 10.0)

memo1 = generate_recommendation(
    test_result=result1,
    srm_result=srm1,
    novelty_result=novelty1,
    practical_result=practical1,
    experiment_name="CTA Button Color & Contrast Test",
    hypothesis="Changing the CTA button to high-contrast blue will increase landing page CVR by 10%+ relative.",
    analyst_notes=(
        "Traffic was evenly split and no SRM detected. No novelty effect observed — "
        "lift is stable across the full experiment window. Recommend immediate full rollout. "
        "Monitor CVR for 2 weeks post-launch to confirm. Estimated annualized impact: "
        "~+13% in phone calls generated from site."
    )
)

print(memo1)
with open('../outputs/memo_scenario1_ship_it.md', 'w') as f:
    f.write(memo1)
print("\nMemo saved to outputs/memo_scenario1_ship_it.md")

# A/B Test Analysis Report
**Experiment:** CTA Button Color & Contrast Test
**Date:** 2026-06-01
**Analyst:** Alex Isabella

---

## Hypothesis
Changing the CTA button to high-contrast blue will increase landing page CVR by 10%+ relative.

---

## Results Summary

| Metric | Value |
|--------|-------|
| Statistical Test Used | Two-Proportion Z-Test |
| Relative Lift | +17.84% |
| P-Value | 0.0055 |
| Statistically Significant | Yes ✅ |
| Practically Significant | Yes ✅ |

---

## Recommendation

### ✅ SHIP IT




---

## Interpretation
✅ Statistically significant. Variant CVR (9.47%) vs Control (8.03%). Relative lift: 17.8% (p=0.0055).

**Novelty Effect:** ✅ No strong novelty effect. Lift appears stable post-warmup.
**SRM Check:** ✅ No SRM detected. Sample split looks healthy.

---

## Analyst Notes
Traffic was evenly split and no SRM detected. No novelty effect observed — lift is stable across the full experiment window. Recommend immediate full rollout. Monitor CVR for 2 weeks post-l

---
## Scenario 2: Headline Copy Test → DO NOT SHIP

In [3]:
s2c, s2v = s2[s2.group=='control'], s2[s2.group=='variant']
result2   = select_and_run_test(control=s2c.revenue_usd.values, variant=s2v.revenue_usd.values, metric_type='continuous')
srm2      = check_sample_ratio_mismatch(len(s2c), len(s2v))
practical2 = assess_practical_significance(
    result2.get('relative_lift_pct', result2.get('median_lift_pct', 0)), 5.0)

memo2 = generate_recommendation(
    test_result=result2,
    srm_result=srm2,
    practical_result=practical2,
    experiment_name="Homepage Headline Copy A/B Test",
    hypothesis="Benefit-focused headline copy will increase revenue per visitor by 5%+ relative.",
    analyst_notes=(
        "No statistically significant difference detected after 28 days and 10,000 users. "
        "The variant headline did not outperform control. Recommend reverting to control "
        "and exploring alternative copy angles. Next iteration: test urgency-based vs "
        "social-proof-based headlines."
    )
)

print(memo2)
with open('../outputs/memo_scenario2_do_not_ship.md', 'w') as f:
    f.write(memo2)
print("\nMemo saved to outputs/memo_scenario2_do_not_ship.md")

# A/B Test Analysis Report
**Experiment:** Homepage Headline Copy A/B Test
**Date:** 2026-06-01
**Analyst:** Alex Isabella

---

## Hypothesis
Benefit-focused headline copy will increase revenue per visitor by 5%+ relative.

---

## Results Summary

| Metric | Value |
|--------|-------|
| Statistical Test Used | Mann-Whitney U Test |
| Relative Lift | -0.60% |
| P-Value | 0.7577 |
| Statistically Significant | No ❌ |
| Practically Significant | No ⚠️ |

---

## Recommendation

### ❌ DO NOT SHIP

**Caveats:**

- Results are not statistically significant. No evidence of a real effect.

---

## Interpretation
❌ Not statistically significant (p=0.7577).


**SRM Check:** ✅ No SRM detected. Sample split looks healthy.

---

## Analyst Notes
No statistically significant difference detected after 28 days and 10,000 users. The variant headline did not outperform control. Recommend reverting to control and exploring alternative copy angles. Next iteration: test urgency-based vs social-proof-base

---
## Scenario 3: Form Redesign → EXTEND EXPERIMENT

In [4]:
s3c, s3v = s3[s3.group=='control'], s3[s3.group=='variant']
result3   = select_and_run_test(metric_type='proportion',
    control_conversions=int(s3c.converted.sum()), control_visitors=len(s3c),
    variant_conversions=int(s3v.converted.sum()), variant_visitors=len(s3v))
srm3      = check_sample_ratio_mismatch(len(s3c), len(s3v))
novelty3  = detect_novelty_effect(s3_daily['control_rate'], s3_daily['variant_rate'])
practical3 = assess_practical_significance(result3['relative_lift_pct'], 5.0)

memo3 = generate_recommendation(
    test_result=result3,
    srm_result=srm3,
    novelty_result=novelty3,
    practical_result=practical3,
    experiment_name="Contact Form Redesign Test",
    hypothesis="Simplified 2-step form layout will increase form completion rate by 5%+.",
    analyst_notes=(
        "Results appear statistically significant at face value, but a novelty effect was "
        "detected in the first 3 days (lift decayed significantly post-warmup). "
        "Steady-state lift is much smaller than early results suggested. "
        "Recommend extending experiment for an additional 14 days to observe true lift. "
        "Do not make a ship decision based on current data."
    )
)

print(memo3)
with open('../outputs/memo_scenario3_extend.md', 'w') as f:
    f.write(memo3)
print("\nMemo saved to outputs/memo_scenario3_extend.md")

# A/B Test Analysis Report
**Experiment:** Contact Form Redesign Test
**Date:** 2026-06-01
**Analyst:** Alex Isabella

---

## Hypothesis
Simplified 2-step form layout will increase form completion rate by 5%+.

---

## Results Summary

| Metric | Value |
|--------|-------|
| Statistical Test Used | Two-Proportion Z-Test |
| Relative Lift | +8.66% |
| P-Value | 0.1619 |
| Statistically Significant | No ❌ |
| Practically Significant | Yes ✅ |

---

## Recommendation

### ⏳ EXTEND EXPERIMENT

**Caveats:**

- Novelty effect flagged (lift decayed 96.5% after warmup). Run longer to observe steady-state behavior.

---

## Interpretation
❌ Not statistically significant (p=0.1619). Cannot conclude the variant outperforms control.

**Novelty Effect:** 🚨 Novelty effect detected — lift decayed 96.5% after warmup period. Steady-state lift is the more reliable signal.
**SRM Check:** ✅ No SRM detected. Sample split looks healthy.

---

## Analyst Notes
Results appear statistically significant at fac

---
## Summary Dashboard

| Experiment | Test Used | Lift | P-Value | Verdict |
|-----------|-----------|------|---------|---------|
| CTA Button Test | Z-Test | ~+21% relative | < 0.001 | ✅ SHIP IT |
| Headline Copy Test | Mann-Whitney | ~+1% | > 0.05 | ❌ DO NOT SHIP |
| Form Redesign | Z-Test | ~+5% apparent | < 0.05 | ⏳ EXTEND |

### Key Takeaways
- Statistical significance alone is not enough — always check practical significance
- A significant result from a broken experiment (SRM) is worthless
- Novelty effects can make bad variants look like winners — always check daily trends
- Clear communication of uncertainty is a sign of a senior analyst, not a weakness

---
*Reports saved in `/outputs/` — copy into Confluence, Notion, or email to stakeholders.*
